## Evaluation & Analysis


# Test Metrics

## Test Data


In [1]:
%load_ext autoreload

Failed to read module file 'C:\Users\LONAL23\AppData\Local\Python\pythoncore-3.14-64\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\LONAL23\OneDrive - PA Consulting Group\7 - Data Scientist\gchq-data-quality\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "c:\Users\LONAL23\OneDrive - PA Consulting Group\7 - Data Scientist\gchq-data-quality\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "C:\Users\LONAL23\AppData\Local\Python\pythoncore-3.14-64\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1398, in _gcd_import
  File "<frozen importlib._bootstra

In [2]:
%autoreload 2
import pandas as pd
from faker import Faker
import uuid
from datetime import datetime, timedelta

# Initialize Faker
fake = Faker()

# Parameters
days = 20
records_per_day = 100
changepoint_day = 10

# Create datetime index (30 days, 100 records per day)
days_range = pd.date_range(start="2024-01-01", periods=days, freq="D")
date_range = days_range.repeat(records_per_day)

# Calculate changepoint date
start_date = datetime(2024, 1, 1)
changepoint_date = start_date + timedelta(days=changepoint_day)

# Generate data
names = []
emails = []

for date in date_range:
    if date < changepoint_date:
        # Before changepoint: use faker
        names.append(fake.name())
        emails.append(fake.email())
    else:
        # After changepoint: UUID for name, truncated email [2:]
        names.append(str(uuid.uuid4()))
        emails.append(fake.email()[4:])  # Remove first 2 characters

# Create DataFrame
df = pd.DataFrame({"name": names, "email": emails}, index=date_range)

df.index.name = "timestamp"
print(f"DataFrame shape: {df.shape}")
print(f"\nChangepoint date: {changepoint_date}")
print(f"\nFirst few rows (day 1):\n{df.head()}")
print(
    f"\nRows at changepoint:\n{df.loc[changepoint_date : changepoint_date + timedelta(hours=1)]}"
)

DataFrame shape: (2000, 2)

Changepoint date: 2024-01-11 00:00:00

First few rows (day 1):
                          name                      email
timestamp                                                
2024-01-01        Donna Martin    barnesandre@example.org
2024-01-01       Lindsey Perry  juliesaunders@example.net
2024-01-01  Christopher Hughes     warrenchad@example.org
2024-01-01         Duane Green        wanda95@example.com
2024-01-01    Matthew Villegas       keykelly@example.com

Rows at changepoint:
                                            name                      email
timestamp                                                                  
2024-01-11  421d6cc8-9bb7-4def-b5ed-882735a256d5          hew44@example.net
2024-01-11  4051d965-6332-4288-9e14-f89b6da4b429     nyroberson@example.com
2024-01-11  b2ea792c-5e84-47b2-91e6-0aa9bef24fb2      cherjimmy@example.org
2024-01-11  c45faec9-03f4-4725-b11d-5de3c33ca706         sallen@example.com
2024-01-11  29242ef1-f3b2

In [12]:
from gchq_data_quality.config import DataQualityConfig
from gchq_data_quality.rules.metrics.string_length import StringLengthRule
from gchq_data_quality.rules.metrics.entropy import EntropyRule
from gchq_data_quality.rules.metrics.punctuation_space_ratio import (
    PunctuationSpaceRatioRule,
)
from gchq_data_quality.rules.metrics.number_ratio import NumberRatioRule
from gchq_data_quality.rules.metrics import CustomMetricRule
from gchq_data_quality import CompletenessRule, UniquenessRule

# Create StringLengthRule rules (powers of 2: 2 to 2048) for name and email
string_length_rules = [
    StringLengthRule(
        field=field,
        threshold=threshold,
        comparison=">=",
        rule_id=f"{field}_length_ge_{threshold}",
    )
    for field in ["name", "email"]
    for threshold in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
]

# Create EntropyRule rules (1 to 8)
entropy_rules = [
    EntropyRule(
        field="email",
        threshold=threshold,
        comparison=">=",
        rule_id=f"email_entropy_ge_{threshold}",
    )
    for threshold in range(1, 9)  # 1 through 8
]

# Create PunctuationSpaceRatioRule rules
punctuation_space_rules = [
    PunctuationSpaceRatioRule(
        field="name",
        threshold=threshold,
        comparison=">=",
        rule_id=f"name_punct_space_ge_{threshold}",
    )
    for threshold in [0.0, 0.2, 0.4, 0.6, 0.8]
] + [
    PunctuationSpaceRatioRule(
        field="email",
        threshold=threshold,
        comparison=">=",
        rule_id=f"email_punct_space_ge_{threshold}",
    )
    for threshold in [0.0, 0.2, 0.4, 0.6, 0.8]
]

# Create NumberRatioRule rules
number_ratio_rules = [
    NumberRatioRule(
        field="name",
        threshold=threshold,
        comparison=">=",
        rule_id=f"name_number_ratio_ge_{threshold}",
    )
    for threshold in [0.0, 0.2, 0.4, 0.6, 0.8]
] + [
    NumberRatioRule(
        field="email",
        threshold=threshold,
        comparison=">=",
        rule_id=f"email_number_ratio_ge_{threshold}",
    )
    for threshold in [0.0, 0.2, 0.4, 0.6, 0.8]
]+ [
    CustomMetricRule(
        field="name",
        metric_name="count_dashes",
        metric_expression="`name`.str.count('-')",
        threshold=threshold,
        rule_id=f"name_custom_metric_ge_{threshold}",
    )
    for threshold in [0.0, 2,4, 8, 10]
]

completeness_rules = [CompletenessRule(field="name"), CompletenessRule(field="email")]
uniqueness_rules = [UniquenessRule(field="name"), UniquenessRule(field="email")]

# Combine all rules
all_rules = (
    string_length_rules
    + entropy_rules
    + punctuation_space_rules
    + number_ratio_rules
    + completeness_rules
    + uniqueness_rules
)

# Create DataQualityConfig
config = DataQualityConfig(dataset_name="test_data_with_changepoint", rules=all_rules)

print(f"Created config with {len(config.rules)} rules:")
print(f"  - {len(string_length_rules)} StringLengthRule rules")
print(f"  - {len(entropy_rules)} EntropyRule rules")
print(f"  - {len(punctuation_space_rules)} PunctuationSpaceRatioRule rules")
print(f"  - {len(number_ratio_rules)} NumberRatioRule rules")
print(f"  - {len(completeness_rules)} CompletenessRule rules")
print(f"  - {len(uniqueness_rules)} UniquenessRule rules")

Created config with 59 rules:
  - 22 StringLengthRule rules
  - 8 EntropyRule rules
  - 10 PunctuationSpaceRatioRule rules
  - 15 NumberRatioRule rules
  - 2 CompletenessRule rules
  - 2 UniquenessRule rules


In [14]:
# Execute the config on the dataframe
ALL_REPORTS = []
for date in df.index.unique():
    print(f"Processing date: {date.date()}")
    config.measurement_time = date
    report = config.execute(df.loc[date])
    ALL_REPORTS.append(report)

# Convert report to dataframe for analysis
report_df = pd.concat(
    [report.to_dataframe() for report in ALL_REPORTS], ignore_index=True
)

Processing date: 2024-01-01
Processing date: 2024-01-02
Processing date: 2024-01-03
Processing date: 2024-01-04
Processing date: 2024-01-05
Processing date: 2024-01-06
Processing date: 2024-01-07
Processing date: 2024-01-08
Processing date: 2024-01-09
Processing date: 2024-01-10
Processing date: 2024-01-11
Processing date: 2024-01-12
Processing date: 2024-01-13
Processing date: 2024-01-14
Processing date: 2024-01-15
Processing date: 2024-01-16
Processing date: 2024-01-17
Processing date: 2024-01-18
Processing date: 2024-01-19
Processing date: 2024-01-20


In [16]:
report_df.query("metric == 'count_dashes'").tail()

,dataset_name,dataset_id,measurement_sample,lifecycle_stage,measurement_time,field,data_quality_dimension,metric,records_evaluated,pass_rate,rule_id,rule_description,rule_data,records_failed_ids,records_failed_sample
1171,test_data_with_changepoint,None,None,None,2024-01-20,name,Metric,count_dashes,100,1.0,name_custom_metric_ge_0.0,None,"{\n ""field"": ""name"",\n ""rule_id"": ""name_cust...",None,None
1172,test_data_with_changepoint,None,None,None,2024-01-20,name,Metric,count_dashes,100,1.0,name_custom_metric_ge_2,None,"{\n ""field"": ""name"",\n ""rule_id"": ""name_cust...",None,None
1173,test_data_with_changepoint,None,None,None,2024-01-20,name,Metric,count_dashes,100,1.0,name_custom_metric_ge_4,None,"{\n ""field"": ""name"",\n ""rule_id"": ""name_cust...",None,None
1174,test_data_with_changepoint,None,None,None,2024-01-20,name,Metric,count_dashes,100,0.0,name_custom_metric_ge_8,None,"{\n ""field"": ""name"",\n ""rule_id"": ""name_cust...","[2024-01-20 00:00:00, 2024-01-20 00:00:00, 202...",[{'name': '3ab82b23-3002-43f6-9d11-84013d9de17...
1175,test_data_with_changepoint,None,None,None,2024-01-20,name,Metric,count_dashes,100,0.0,name_custom_metric_ge_10,None,"{\n ""field"": ""name"",\n ""rule_id"": ""name_cust...","[2024-01-20 00:00:00, 2024-01-20 00:00:00, 202...",[{'name': '3ab82b23-3002-43f6-9d11-84013d9de17...


In [17]:
import matplotlib.pyplot as plt
import plotly.express as px

# Group by measurement_time and data_quality_dimension to get average pass_rate
time_series_results = (
    report_df.groupby(["measurement_time", "data_quality_dimension"])
    .agg({"pass_rate": "mean"})
    .reset_index()
    .sort_values("measurement_time")
)

# Create interactive Plotly line chart
fig = px.line(
    time_series_results,
    x="measurement_time",
    y="pass_rate",
    color="data_quality_dimension",
    title="Average Pass Rate Over Time by Data Quality Dimension",
    labels={
        "measurement_time": "Time",
        "pass_rate": "Average Pass Rate",
        "data_quality_dimension": "Data Quality Dimension",
    },
    markers=True,
    height=600,
)

fig.update_layout(
    hovermode="x unified", yaxis=dict(range=[0, 1]), template="plotly_white"
)

fig.show()

# Print summary statistics
print("\nSummary by Data Quality Dimension:")
print(
    time_series_results.groupby("data_quality_dimension")["pass_rate"].agg(
        ["mean", "min", "max"]
    )
)


Summary by Data Quality Dimension:
                            mean       min       max
data_quality_dimension                              
Completeness            1.000000  1.000000  1.000000
Metric                  0.329609  0.274909  0.384182
Uniqueness              0.998000  0.990000  1.000000


In [18]:
# Filter for Metric dimension and group by field and metric type
metric_results = (
    report_df[report_df["data_quality_dimension"] == "Metric"]
    .groupby(["measurement_time", "field", "metric"])
    .agg({"pass_rate": "mean"})
    .reset_index()
    .sort_values(["measurement_time", "field", "metric"])
)

# Separate Metric charts for name and email
metric_name = metric_results[metric_results["field"] == "name"]
metric_email = metric_results[metric_results["field"] == "email"]

fig_metric_name = px.line(
    metric_name,
    x="measurement_time",
    y="pass_rate",
    color="metric",
    title="Metric Dimension Pass Rate Over Time for name",
    labels={
        "measurement_time": "Time",
        "pass_rate": "Average Pass Rate",
        "metric": "Metric Type",
    },
    markers=True,
    height=500,
)
fig_metric_name.update_layout(
    hovermode="x unified", yaxis=dict(range=[0, 1]), template="plotly_white"
)

fig_metric_email = px.line(
    metric_email,
    x="measurement_time",
    y="pass_rate",
    color="metric",
    title="Metric Dimension Pass Rate Over Time for email",
    labels={
        "measurement_time": "Time",
        "pass_rate": "Average Pass Rate",
        "metric": "Metric Type",
    },
    markers=True,
    height=500,
)
fig_metric_email.update_layout(
    hovermode="x unified", yaxis=dict(range=[0, 1]), template="plotly_white"
)

fig_metric_name.show()
fig_metric_email.show()

# Print summary statistics for Metric dimension by field and metric type
print("\nSummary by Field and Metric Type:")
print(
    metric_results.groupby(["field", "metric"])["pass_rate"].agg(["mean", "min", "max"])
)


Summary by Field and Metric Type:
                                   mean       min       max
field metric                                               
email entropy                  0.378938  0.373750  0.387500
      number_ratio             0.200000  0.200000  0.200000
      punctuation_space_ratio  0.200000  0.200000  0.200000
      string_length            0.351818  0.334545  0.363636
name  count_dashes             0.400000  0.200000  0.600000
      number_ratio             0.430200  0.200000  0.674000
      punctuation_space_ratio  0.201000  0.200000  0.208000
      string_length            0.370091  0.279091  0.454545
